In [ ]:
import os 
import corner
import sys
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = 'serif'
plt.rcParams["mathtext.fontset"] = "cm"
import json

#import pandas as pd
import h5py
import emcee
import matplotlib.lines as mlines

In [ ]:
# Set here your local path to data folder


path_to_data = 'results/'

In [ ]:
names =  { "a":r'$a$',
            "b":r'$a$',
            "c":r'$a$'
          "d":r'$a$'
}


priorLimits =  {"a":(0, 1),
            "b":(0, 1),
            "c":(0, 1),
          "d":(0, 1),}

In [ ]:
def get_all_samples(data_path,  thin=None, ntaus_burnin=2, ntaus_thin=1, burnin=100, ):

    
    
    chain_path=os.path.join(data_path , 'chains.h5')
    
    all_samples = []
    all_logprob_samples = []
    
    i=0
    while i<1000:
        if os.path.exists(chain_path):
            print('Reading chains from %s...'%chain_path)
            with h5py.File(chain_path, "r") as hf:
                all_samples_ = np.copy(hf['samples'])
                all_logprob_samples_ = np.copy(hf['logprob'])
            print("samples shape : {0}".format(all_samples_.shape))
            all_samples.append(all_samples_)
            all_logprob_samples.append(all_logprob_samples_)
        i+=1
        chain_path=os.path.join(data_path , 'chains_run'+str(i)+'.h5')
      
    all_samples = np.concatenate(all_samples)
    all_logprob_samples = np.concatenate(all_logprob_samples)
    
    print("all samples shape : {0}".format(all_samples.shape))
    print("all prob shape : {0}".format(all_logprob_samples.shape))
    
    print('Computing autocorrelation...')
    tau = emcee.autocorr.integrated_time(all_samples, c=10, tol=50, quiet=True, )
    #tau = reader.get_autocorr_time(quiet=True)
    
    try:
        burnin = int(ntaus_burnin * np.max(tau))
    except ValueError:
        burnin=burnin
    if thin is None:
        try:
            thin = int(ntaus_thin * np.min(tau))
        except ValueError:
            thin=thin #int(burnin/2)
    if burnin/2<1:
        burnin=0
        thin=1
    
    print("tau: {0}".format(tau))
    print("tau max: {0}".format(np.max(tau)))
    print("burn-in: {0}".format(burnin))
    print("thin: {0}".format(thin))
    
    #if not is_1D:
    
    samples_clean = all_samples[burnin::thin, :, :]
    print("Samples clean shape : {0}".format(samples_clean.shape))
    

    samples = samples_clean.reshape( samples_clean.shape[0]*samples_clean.shape[1],samples_clean.shape[2] )
    weights = all_logprob_samples[burnin::thin, :].reshape( all_logprob_samples[burnin::thin, :].shape[0]*all_logprob_samples[burnin::thin, :].shape[1])
    
    
    print("flat chains shape (after discarding burnin phase and thin): {0}".format(samples.shape))

    return tau, all_samples,  samples, burnin, weights

### Read chains

In [ ]:
tau, all_samples, samples, burnin, weights = get_all_samples(path_to_data, 
                                                                      ntaus_burnin=2, 
                                                                      ntaus_thin=1 )

### Plot

In [ ]:
with open( os.path.join(path_to_data, 'settings_original.json'), 'r') as fp: # import settings for the analysis 
        settings = json.load(fp)

In [ ]:
vars_plot = settings['params_inference']
true_vals = settings['fiducial_vals']

In [ ]:
labels=[names[n] for n in vars_plot]
ndim=len(vars_plot)

truths = [ true_vals[n] for n in vars_plot ]

smooth=0.5
hist_bin_factor=1




In [ ]:
figure = corner.corner(samples, 
                    labels=labels,  
                    truths=truths,
                    levels=[0.68, 0.90],
                    show_titles=True, 
                    title_kwargs={"fontsize": 15, "fontweight":"bold",}, label_kwargs={"fontsize": 20},
                    density=True,
                    smooth=smooth,              
                    color= 'darkred', 
                    verbose=False, 
                    plot_datapoints=True, 
                    fill_contours=True,
                    bins=15, 
                       title_fmt='.3f', 
                  hist_bin_factor=hist_bin_factor,
                    quantiles=None, #[0.05, 0.95],                    
                   );